# **Natural Language Processing - Assignment 3: Transformer Fine-tuning + Robustness + Limitations**

### Imports

In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from datasets import load_dataset

### Load Dataset

In [12]:
device = torch.device("cuda") if torch.cuda.is_available() else "cpu"

raw = load_dataset("sh0416/ag_news")

test_set = raw["test"].shuffle(seed=13).select(range(100))

split = raw["train"].train_test_split(test_size=0.1, seed=13)

train_set = (
    split["train"].shuffle(seed=13).select(range(100))
)  # Using a subset for quick fine-tuning

val_set = split["test"].shuffle(seed=13).select(range(100))


### Tokenization

In [13]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize_function(examples):
    # Combining title and description for tokenization
    combined_texts = [
        t + " " + d for t, d in zip(examples["title"], examples["description"])
    ]
    tokenized_inputs = tokenizer(combined_texts, padding="max_length", truncation=True)
    # Add labels to the tokenized inputs and adjust them to be 0-indexed
    tokenized_inputs["labels"] = [label - 1 for label in examples["label"]]
    return tokenized_inputs


tokenized_train = train_set.map(tokenize_function, batched=True)
tokenized_val = val_set.map(tokenize_function, batched=True)
tokenized_test = test_set.map(tokenize_function, batched=True)


Map: 100%|██████████| 100/100 [00:00<00:00, 1839.70 examples/s]


### Loading the Pre-trained Model, Setting Up the Trainer, and Training

In [15]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4).to(
    device
)

training_args = TrainingArguments(
    eval_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

trainer.train()


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3347.36it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

### Evaluation and Testing

In [ ]:
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

# input_string = "I really liked this tutorial!"

# # Tokenize the input string
# inputs = tokenizer(input_string, return_tensors="pt").to(device)

# # Get predictions (logits)
# with torch.no_grad():  # Disable gradient computation since we're just doing inference
#     outputs = model(**inputs)
#     logits = outputs.logits

# predicted_label = torch.argmax(logits, dim=1).item()


# print(f"Predicted label: {predicted_label}")
